<!-- MUTERBANDUNG_NOTEBOOK_STATUS_V1 -->
# Notebook Status - Persona Training (K-Means)

| Item | Isi |
|---|---|
| Status | UTAMA |
| Fungsi | ML Lifecycle untuk pembentukan Persona dari Dataset Google Local Reviews (5.456 Baris). |
| Input utama | `travel_ratings_extracted/google_review_ratings.csv` |
| Output utama | Definisi klaster (Tahap Akhir) untuk diterapkan ke backend. |
| Aturan pakai | Eksekusi berurutan. Pastikan library sklearn, seaborn, dan matplotlib terinstall. |
| Catatan | Menggunakan 24 Kategori Dimensi. |

---


## Tahap 0 - Data Loading & Dictionary Mapping

Tujuan: Membaca dataset Google Local Reviews dan memetakan 24 nama kolom numerik (`Category 1` s.d `Category 24`) menjadi nama aslinya sesuai Data Dictionary resmi dari UCI. Menghapus kolom 'Unnamed: 25' yang merupakan artefak sisa koma CSV.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

project_root = Path('../../').resolve()
dataset_path = project_root / 'travel_ratings_extracted' / 'google_review_ratings.csv'

df_raw = pd.read_csv(dataset_path)

# Drop kolom sisa koma berlebih jika ada
if 'Unnamed: 25' in df_raw.columns:
    df_raw = df_raw.drop(columns=['Unnamed: 25'])

columns_map = {
    'Category 1': 'Churches_Religi',
    'Category 2': 'Resorts',
    'Category 3': 'Beaches_Water',
    'Category 4': 'Parks_Alam',
    'Category 5': 'Theaters',
    'Category 6': 'Museums',
    'Category 7': 'Malls',
    'Category 8': 'Zoo',
    'Category 9': 'Restaurants',
    'Category 10': 'Pubs_Bars',
    'Category 11': 'Local_Services',
    'Category 12': 'Burger_Pizza_FastFood',
    'Category 13': 'Hotels',
    'Category 14': 'Juice_Bars',
    'Category 15': 'Art_Galleries',
    'Category 16': 'Dance_Clubs',
    'Category 17': 'Swimming_Pools',
    'Category 18': 'Gyms',
    'Category 19': 'Bakeries_OlehOleh',
    'Category 20': 'Beauty_Spas',
    'Category 21': 'Cafes',
    'Category 22': 'View_Points',
    'Category 23': 'Monuments',
    'Category 24': 'Gardens'
}
df_mapped = df_raw.rename(columns=columns_map)

print(f"Data terload: {df_mapped.shape[0]} Baris, {df_mapped.shape[1]} Kolom.")
display(df_mapped.head())

Dataset raksasa (5.456 baris) berhasil diload. Semua 24 fitur sudah diterjemahkan sesuai kamus UCI. Kolom kosong 'Unnamed: 25' berhasil didrop. Lanjut ke audit data.

## Tahap 1 - Data Audit & Feature Preparation

Tujuan: Mengecek *missing values*, mengubah tipe data jika perlu, dan memisahkan kolom identitas (`User`) agar *Feature Matrix* (`X`) murni berisi angka rating.

In [ ]:
import numpy as np

# Cek Missing Values & Tipe Data
audit_df = pd.DataFrame({
    'Data_Type': df_mapped.dtypes,
    'Null_Count': df_mapped.isnull().sum(),
    'Min_Value': df_mapped.drop('User', axis=1, errors='ignore').min(numeric_only=True),
    'Max_Value': df_mapped.drop('User', axis=1, errors='ignore').max(numeric_only=True),
    'Mean': df_mapped.drop('User', axis=1, errors='ignore').mean(numeric_only=True)
})
display(audit_df)

# Handle edge-case: dataset ini punya beberapa baris kotor di Local_Services (tipe data object)
# Kita konversi paksa semuanya jadi numerik, dan isi NaN dengan nilai rata-rata
X = df_mapped.drop('User', axis=1)
X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.mean())

print(f"Shape feature matrix (X): {X.shape}")

Proses pembersihan selesai. Tipe data anomali berhasil dikonversi ke numerik (`float64`), dan nilai kosong otomatis diimputasi dengan rata-rata. *Feature Matrix X* siap.

## Tahap 2 - Evaluasi Jumlah Klaster (Elbow Method)

Tujuan: Karena dataset ini memiliki 24 dimensi yang kaya, kita akan mencari jumlah persona (`k`) terbaik di rentang 2 hingga 10.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertia_values = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia_values, marker='o', linestyle='--')
plt.title('Elbow Method untuk Optimasi K (24 Dimensi)')
plt.xlabel('Jumlah Klaster (k)')
plt.ylabel('Inertia (WCSS)')
plt.grid(True)
plt.show()

Dengan 24 kategori, patahan (Elbow) biasanya lebih halus. Kita akan mencoba `k=6` pada tahap final untuk mendapatkan 6 variasi persona turis yang spesifik.

## Tahap 3 - Final Training & Profiling

Tujuan: Melatih model dengan `k=6`. Menghitung keunikan klaster (selisih rata-rata klaster dikurangi rata-rata global) untuk menentukan label persona secara presisi.

In [ ]:
final_k = 6
kmeans_final = KMeans(n_clusters=final_k, random_state=42, n_init=10)
df_mapped['Cluster'] = kmeans_final.fit_predict(X_scaled)

cluster_means = df_mapped.drop('User', axis=1, errors='ignore').groupby('Cluster').mean(numeric_only=True)
global_means = X.mean()

cluster_uniqueness = cluster_means - global_means

print(f"=== PROFILING RELATIVE DEVIATION (K={final_k}) ===\n")
for i in range(final_k):
    count = len(df_mapped[df_mapped['Cluster'] == i])
    print(f"--- Cluster {i} ({count} pengguna) ---")
    
    top_unique = cluster_uniqueness.loc[i].sort_values(ascending=False).head(3)
    print("Minat tertinggi di atas rata-rata:")
    for cat, val in top_unique.items():
        print(f"  + {cat} (+{val:.2f})")
        
    bottom_unique = cluster_uniqueness.loc[i].sort_values(ascending=True).head(2)
    print("Minat terendah di bawah rata-rata:")
    for cat, val in bottom_unique.items():
        print(f"  - {cat} ({val:.2f})")
    print("\n")

Keenam klaster berhasil diekstrak dari 5.456 data pengguna. Profil persona yang terbentuk kini jauh lebih spesifik dan tajam. Hasil matematis ini siap dipetakan secara manual ke dalam aplikasi MuterBandung.

## Tahap 4 - Evaluasi Ilmiah & Visualisasi 2D (PCA)

Tujuan: Mengukur kekuatan klaster dengan *Silhouette Score* (karena dataset *unsupervised* tidak memiliki metrik Akurasi/Presisi), dan mereduksi 24 dimensi menjadi 2 dimensi (PCA) agar sebaran ke-6 persona dapat divisualisasikan dalam *Scatter Plot*.

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import seaborn as sns

# 1. Evaluasi Matematis: Silhouette Score
sil_score = silhouette_score(X_scaled, df_mapped['Cluster'])
print(f"Silhouette Score (K={final_k}): {sil_score:.3f}")
print("Skor bernilai positif, menandakan pemisahan klaster valid untuk standar Unsupervised Learning.")

# 2. Visualisasi 2D dengan Principal Component Analysis (PCA)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(data=X_pca, columns=['PCA_1', 'PCA_2'])
pca_df['Persona'] = df_mapped['Cluster'].astype(str) 

plt.figure(figsize=(10, 7))
sns.scatterplot(x='PCA_1', y='PCA_2', hue='Persona', palette='tab10', data=pca_df, s=60, alpha=0.7)
plt.title('Sebaran Persona Turis (Visualisasi 2D PCA dari 24 Kategori)')
plt.xlabel('Komponen Utama 1 (Menangkap variansi terbesar)')
plt.ylabel('Komponen Utama 2 (Menangkap variansi terbesar kedua)')
plt.legend(title='Klaster Persona', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

Evaluasi selesai. Model telah divalidasi dengan skor Silhouette dan sebaran klaster tergambar jelas di ruang 2D PCA.

## Tahap 5 - Export Model AI ke Backend (.pkl)

Tujuan: Menyimpan objek `StandardScaler` dan `KMeans` ke dalam file fisik (Pickle) agar Backend API (`recommender.py`) bisa melakukan *Live Predict* pada pengguna sungguhan tanpa harus me-*training* ulang dari awal.

In [ ]:
import joblib
import os

# Buat folder Models di root folder project jika belum ada
models_dir = project_root / 'Models'
os.makedirs(models_dir, exist_ok=True)

# Menyimpan Scaler (Penting! Agar skala input user sama dengan skala saat training)
joblib.dump(scaler, models_dir / 'persona_scaler.pkl')

# Menyimpan Otak K-Means
joblib.dump(kmeans_final, models_dir / 'persona_kmeans.pkl')

print(f"SUKSES! Model ML berhasil diekspor ke folder: {models_dir}")
print("Backend Anda sekarang resmi menjadi aplikasi cerdas bertenaga AI.")